# The pooling problem: when a quality is an unknown times an unknown

Three crude sources with different sulfur contents and prices. Two of them can only be delivered
through a shared pool, where they mix; the third goes straight to the customers. Two products, each
with a sulfur cap and a demand cap and a price. Decide the flows to make the most money.

If the sulfur content of the pool were known, this would be an LP. It is not known: it depends on
how much of each source went in, which is what you are deciding. So the sulfur balance at the pool
multiplies a quality by a flow — two unknowns — and the model is **bilinear**. That single product
is what makes the problem nonconvex, and is the reason Haverly's example from 1978 is still the
standard test of a global solver.

## Setup: where the package lives

This notebook builds its models by hand and then checks them against `orteach`, the package in
`src/`. Run from a clone of the repository, `../../src` is right there. On Colab there is no clone
until this cell makes one, and no `gurobipy` until it installs it. Nothing here needs a secret.

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/sear-labs/teaching-code"

try:
    import google.colab                      # noqa: F401 - succeeds only on Colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if not os.path.isdir("/content/teaching-code"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/teaching-code"], check=True)
    os.chdir("/content/teaching-code/notebooks/06_nonlinear_and_convex")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gurobipy>=11,<14"], check=True)

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
try:
    import orteach                            # noqa: F401
except ImportError:
    raise SystemExit("orteach not found: run this notebook from its own folder inside the repository, "
                     "so that ../../src exists.")
root = os.path.abspath(os.path.join("..", ".."))
print("package:", os.path.relpath(os.path.dirname(orteach.__file__), root))

package: src\orteach


## Licence setup

Nothing here needs a key: `pip install gurobipy` ships a size-limited licence and the models below
sit well inside it. A machine with its own licence file uses that instead, and on Colab three
secrets read from the key icon in the left sidebar are used when they are there — three named here,
none contained. The environment starts silent, so no licence number lands in an output cell.

In [2]:
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
        licence = "Colab Secrets (WLS)"
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        licence = "the size-limited licence pip ships"     # no key needed; see the note above
except ImportError:
    licence = "local gurobi.lic"
env.start()
print("licence:", licence)

licence: local gurobi.lic


## Two tables

Sources — sulfur, price, and whether each feeds the pool or goes direct — and products — price,
sulfur cap, demand cap. Instance data indexed by the model's sets, so both live in `data/raw/` and
both this notebook and the package read them. Sulfur is stored in percent and loaded as a fraction.

In [3]:
from orteach import tolerance
from orteach.nonconvex import load_pooling

inst = load_pooling()

print(f"{'source':7} {'sulfur':>7} {'$/unit':>7}  route")
for s in inst.sources:
    print(f"{s:7} {inst.sulfur[s]:7.3f} {inst.cost[s]:7.0f}  {inst.route[s]}")
print(f"\n{'product':8} {'$/unit':>7} {'max S':>7} {'demand':>7}")
for p in inst.products:
    print(f"{p:8} {inst.price[p]:7.0f} {inst.max_sulfur[p]:7.3f} {inst.demand[p]:7.0f}")
print()
print("pooled:", inst.pooled, "  direct:", inst.direct, "  sulfur['A'] =", inst.sulfur["A"])

# to try a different price, edit the loaded table; the change reaches the package check at the bottom:
# inst.price["X"] = 12.0

source   sulfur  $/unit  route
A         0.030       6  pool
B         0.010      16  pool
C         0.020      10  direct

product   $/unit   max S  demand
X              9   0.025     100
Y             15   0.015     200

pooled: ['A', 'B']   direct: ['C']   sulfur['A'] = 0.03


## Predict before building anything

Product Y pays more than X and demands more, but its sulfur cap is tighter. Source A is the cheapest
and the dirtiest; B the cleanest and the dearest; C is in between and skips the pool. Write down which
product you would make, out of which sources, and roughly what the profit would be — then build the
model.

## Flows, with boxes

One flow per pooled source into the pool, one per product out of the pool, one per (direct source,
product). Every flow gets a finite upper bound, because spatial branch-and-bound builds its convex
envelopes over boxes and an unbounded bilinear term gives it nothing to build over. Work out the
smallest box that cannot bind before reading the next cell: what is the most any single flow could
ever carry?

In [4]:
# Total demand: nothing sold exceeds a product's demand, so no one flow can exceed their sum.
# The source used 250 - above the LARGEST demand but below the sum, so it was inert at the shipped
# prices and would not have been at others.
FLOW_CAP = sum(inst.demand.values())

m = gp.Model("pooling", env=env)
tolerance.apply(m)

to_pool   = m.addVars(inst.pooled, lb=0.0, ub=FLOW_CAP, name="to_pool")
from_pool = m.addVars(inst.products, lb=0.0, ub=FLOW_CAP, name="from_pool")
direct    = m.addVars([(s, p) for s in inst.direct for p in inst.products], lb=0.0, ub=FLOW_CAP, name="direct")
m.update()
print(m.NumVars, "flow variables:", list(to_pool.keys()), list(from_pool.keys()), list(direct.keys()))

6 flow variables: ['A', 'B'] ['X', 'Y'] [('C', 'X'), ('C', 'Y')]


## The quality variables — where the model stops being linear

The pool's sulfur fraction $q_P$ and each product's delivered fraction $q_j$ are unknowns. The pool's
sulfur balance says *sulfur in equals sulfur out*: the fraction times the total flow through the
pool equals the sulfur the sources brought. $q_P \times$ flow is the bilinear term. There are other
ways to write this model — one of them is the third exercise at the bottom — and every one of them
still multiplies two unknowns somewhere. Before running the cell, say where the product would go if
you eliminated $q_P$.

In [5]:
q_pool = m.addVar(lb=0.0, ub=1.0, name="pool_sulfur")
q_prod = m.addVars(inst.products, lb=0.0, ub=1.0, name="product_sulfur")

pool_balance = m.addConstr(to_pool.sum() == from_pool.sum(), name="pool_balance")
pool_quality = m.addConstr(q_pool * to_pool.sum() == gp.quicksum(inst.sulfur[s] * to_pool[s] for s in inst.pooled),
                           name="pool_quality")
m.update()
print("pool balance :", m.getRow(pool_balance), "=", pool_balance.RHS)
print("pool quality :", m.getQCRow(pool_quality), "=", pool_quality.QCRHS)

pool balance : to_pool[A] + to_pool[B] + -1.0 from_pool[X] + -1.0 from_pool[Y] = 0.0
pool quality : -0.03 to_pool[A] + -0.01 to_pool[B] + [ to_pool[A] * pool_sulfur + to_pool[B] * pool_sulfur ] = 0.0


The same balance at each product: what it receives from the pool carries the pool's fraction, what it
receives direct carries that source's known fraction, and the delivered fraction must be under the
cap. Demand is a ceiling, not a requirement — the model may make less than the market takes.

In [6]:
for p in inst.products:
    inflow = from_pool[p] + direct.sum("*", p)
    m.addConstr(q_prod[p] * inflow == q_pool * from_pool[p] + gp.quicksum(inst.sulfur[s] * direct[s, p] for s in inst.direct),
                name=f"quality[{p}]")
    m.addConstr(inflow <= inst.demand[p], name=f"demand[{p}]")
    m.addConstr(q_prod[p] <= inst.max_sulfur[p], name=f"spec[{p}]")
m.update()
print(m.NumConstrs, "linear rows,", m.NumQConstrs, "bilinear rows")

5 linear rows, 3 bilinear rows


## The objective: revenue minus purchases

In [7]:
revenue = gp.quicksum(inst.price[p] * (from_pool[p] + direct.sum("*", p)) for p in inst.products)
purchase = gp.quicksum(inst.cost[s] * to_pool[s] for s in inst.pooled) \
    + gp.quicksum(inst.cost[s] * direct[s, p] for s in inst.direct for p in inst.products)
m.setObjective(revenue - purchase, GRB.MAXIMIZE)
m.update()
print(m.getObjective())

-6.0 to_pool[A] + -16.0 to_pool[B] + 9.0 from_pool[X] + 15.0 from_pool[Y] + -1.0 direct[C,X] + 5.0 direct[C,Y]


## Ask the solver as if the model were convex

Gurobi checks the shape of every quadratic row before it starts. Tell it, with `NonConvex = 0`, that
you expect a convex model, and see what it says about the bilinear rows.

In [8]:
m.Params.NonConvex = 0
try:
    m.optimize()
    print("status", m.Status)
except gp.GurobiError as e:
    print("GurobiError:", e)

GurobiError: Quadratic equality constraints are non-convex. Set NonConvex parameter to -1 or 2 to solve model.


## Now let it branch

`NonConvex = 2` turns on spatial branch-and-bound: the solver relaxes each product of variables to a
convex envelope over its box, solves, splits a box, and repeats until the bound meets the best
solution found. Predict the profit before running it, and which product it makes.

In [9]:
m.Params.NonConvex = 2
m.optimize()
profit_hand = m.ObjVal
print(f"status {m.Status}   profit {profit_hand:.2f}   proven bound {m.ObjBound:.2f}")
print()
for s in inst.pooled:
    print(f"  {s} -> pool      {to_pool[s].X:7.2f}")
for p in inst.products:
    print(f"  pool -> {p}      {from_pool[p].X:7.2f}")
for (s, p), v in direct.items():
    print(f"  {s} -> {p} direct {v.X:7.2f}")
made = {p: from_pool[p].X + sum(direct[s, p].X for s in inst.direct) for p in inst.products}
print(f"\npool sulfur {q_pool.X:.4f}   delivered: " + "  ".join(
    f"{p} {q_prod[p].X:.4f} (cap {inst.max_sulfur[p]:.3f})" if made[p] > tolerance.FEASIBILITY_ATOL else f"{p} not made"
    for p in inst.products))

status 2   profit 400.00   proven bound 400.00

  A -> pool         0.00
  B -> pool       100.00
  pool -> X         0.00
  pool -> Y       100.00
  C -> X direct    0.00
  C -> Y direct  100.00

pool sulfur 0.0100   delivered: X not made  Y 0.0150 (cap 0.015)


A product that is not made has no delivered fraction: its quality row reads $q_j \times 0 = 0$, so
the variable is free to sit anywhere and its value means nothing. That is why it is not printed —
and, at the bottom, why it is not compared.

## Check the qualities from the flows alone

The quality rows are the part of the model most easily written wrong, so recompute every delivered
sulfur fraction from the flows — no quality variables, no solver — and compare.

In [10]:
flows_pool = {s: to_pool[s].X for s in inst.pooled}
pool_in = sum(flows_pool.values())
pool_frac = sum(inst.sulfur[s] * f for s, f in flows_pool.items()) / pool_in if pool_in > tolerance.FEASIBILITY_ATOL else 0.0
print(f"pool sulfur from flows {pool_frac:.4f}   variable said {q_pool.X:.4f}")
for p in inst.products:
    total = from_pool[p].X + sum(direct[s, p].X for s in inst.direct)
    if total > tolerance.FEASIBILITY_ATOL:
        got = (pool_frac * from_pool[p].X + sum(inst.sulfur[s] * direct[s, p].X for s in inst.direct)) / total
        print(f"{p}: {total:6.1f} units at sulfur {got:.4f} from flows   variable said {q_prod[p].X:.4f}   cap {inst.max_sulfur[p]:.3f}")
    else:
        print(f"{p}: nothing made")

pool sulfur from flows 0.0100   variable said 0.0100
X: nothing made
Y:  200.0 units at sulfur 0.0150 from flows   variable said 0.0150   cap 0.015


---

## The landscape behind the bilinear row

Pin the pool's sulfur fraction at a few values and re-solve each time. With $q_P$ fixed the model
tells you the best you can do *given that pool*, and the profits trace the landscape a downhill
method would have to cross. Predict the shape before running: one hump, or two?

In [11]:
landscape = {}
for fixed in (0.010, 0.015, 0.020, 0.025, 0.030):
    q_pool.LB = q_pool.UB = fixed
    m.optimize()
    landscape[fixed] = round(m.ObjVal, 6) + 0.0          # + 0.0 turns a solver's -0.0 into 0.0
    print(f"pool sulfur fixed at {fixed:.3f}:  profit {landscape[fixed]:8.2f}   "
          f"X {from_pool['X'].X + sum(direct[s, 'X'].X for s in inst.direct):6.1f}   "
          f"Y {from_pool['Y'].X + sum(direct[s, 'Y'].X for s in inst.direct):6.1f}")
q_pool.LB, q_pool.UB = 0.0, 1.0          # release the pin
m.optimize()
print(f"\npin released, re-solved: profit {m.ObjVal:8.2f}   pool sulfur {q_pool.X:.4f}"
      f"   <- back to the free optimum, and the values the check below reads")

pool sulfur fixed at 0.010:  profit   400.00   X    0.0   Y  200.0
pool sulfur fixed at 0.015:  profit   300.00   X    0.0   Y  200.0
pool sulfur fixed at 0.020:  profit     0.00   X    0.0   Y    0.0
pool sulfur fixed at 0.025:  profit    50.00   X  100.0   Y    0.0
pool sulfur fixed at 0.030:  profit   100.00   X  100.0   Y    0.0

pin released, re-solved: profit   400.00   pool sulfur 0.0100   <- back to the free optimum, and the values the check below reads


Two profitable regions with a valley between them, and the run above ends back at the free optimum.
What does each hump correspond to in terms of which product the pool serves? If a downhill method
were started at the right-hand one, what would it report, and what would it have to be given in order
to know there was anything better?

---

# Now the streamlined version

One bilinear model built by hand and solved six times, so the package owns it now:
`solve_pooling` takes the two tables and the box as arguments, `profit_of` recomputes the objective
from flows, and `delivered_sulfur` does the check above.

In [12]:
from orteach import nonconvex as nc
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg = nc.solve_pooling(inst, flow_cap=FLOW_CAP, env=env)
pkg_landscape = nc.pooling_landscape(inst, list(landscape), flow_cap=FLOW_CAP, env=env)
print(f"{pkg.label}: profit {pkg.objective:.2f}   bound {pkg.bound:.2f}   pool sulfur {pkg.pool_sulfur:.4f}")
print("profit from flows:", round(nc.profit_of(inst, pkg), 6))
# only for products that are actually made: an unmade one has no delivered fraction to report
print("delivered sulfur :", {p: round(nc.delivered_sulfur(inst, pkg, p), 4)
                             for p in inst.products if pkg.sold(p) > tolerance.FEASIBILITY_ATOL})
print("landscape        :", pkg_landscape)

Gurobi, P-formulation: profit 400.00   bound 400.00   pool sulfur 0.0100
profit from flows: 400.0
delivered sulfur : {'Y': 0.015}
landscape        : {0.01: 400.0, 0.015: 300.0, 0.02: -0.0, 0.025: 50.0, 0.03: 100.0}


## The agreement assertion

The hand-built model against the package, number by number: profit, proven bound, every flow, the
pool's fraction, the delivered fraction of every product that is made, and every point of the
landscape sweep. The optimum here is
unique — the cheapest way to fill Y's demand at exactly its cap is one specific blend, and X cannot
be made at a profit — so comparing flows is legitimate. A product that is not made has an undefined
quality variable on both sides, and comparing two undefined numbers would test nothing but the
solver's path, so those are left out. Both sides solved at the same tolerances with the same boxes.

In [13]:
checks = [("profit", profit_hand, pkg.objective),
          ("proven bound", m.ObjBound, pkg.bound),
          ("pool sulfur", q_pool.X, pkg.pool_sulfur)]
for s in inst.pooled:
    checks.append((f"{s} -> pool", to_pool[s].X, pkg.to_pool[s]))
for p in inst.products:
    checks.append((f"pool -> {p}", from_pool[p].X, pkg.from_pool[p]))
    if made[p] > tolerance.FEASIBILITY_ATOL:
        checks.append((f"delivered sulfur {p}", q_prod[p].X, pkg.product_sulfur[p]))
for (s, p) in direct:
    checks.append((f"{s} -> {p}", direct[s, p].X, pkg.direct[s, p]))
for fixed, profit in landscape.items():
    checks.append((f"landscape at {fixed:.3f}", profit, pkg_landscape[fixed]))

worst = max(rel_diff(h, k) for _, h, k in checks)
print(f"{len(checks)} comparisons")
for name, hand, packaged in checks[:4]:
    print(f"  {name:20} hand {hand:10.4f}   package {packaged:10.4f}   rel {rel_diff(hand, packaged):.2e}")
print("  ...")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

15 comparisons
  profit               hand   400.0000   package   400.0000   rel 0.00e+00
  proven bound         hand   400.0000   package   400.0000   rel 0.00e+00
  pool sulfur          hand     0.0100   package     0.0100   rel 0.00e+00
  A -> pool            hand     0.0000   package     0.0000   rel 0.00e+00
  ...

notebook and package agree to 0.0e+00


---

## Where to take this next

- Raise X's price until the solver switches humps. At what price does it happen, and what does the
  pool's sulfur fraction do at the switch?
- Add a second pool that only C can enter and that only Y can leave. Is the answer any different, and
  what did the extra structure cost in bilinear rows?
- The quality variables can be eliminated: write each product's spec directly as
  $q_P\,f_{P,j} + \sum_i s_i f_{i,j} \le \bar{s}_j\,(\text{inflow}_j)$. Rebuild it that way. Fewer
  variables, the same bilinear terms — does the proof get faster?